# TIA Portal RAG 模板库使用演示

本 Notebook 演示 `RAG/rag_retriever.py` 模块的完整使用流程：

1. 扫描并列出所有可用模板
2. 按关键词检索最相关模板
3. 查看模板接口结构（摘要 / 完整）
4. 扩充模板库的方法

> **运行前提**：在 `SCDW/` 根目录下运行本 Notebook（确保 Python 能找到 `RAG/` 目录）

In [ ]:
import sys
import os

# 将 RAG 目录加入路径
rag_dir = os.path.join(os.getcwd(), 'RAG')
if rag_dir not in sys.path:
    sys.path.insert(0, rag_dir)

from rag_retriever import (
    TemplateLibrary,
    list_templates,
    search_templates,
    get_template_xml,
)

print('RAG 模块加载成功 ✅')

## 1. 列出所有可用模板

In [ ]:
templates = list_templates()
print(f'共找到 {len(templates)} 个模板：\n')
for t in templates:
    print(f"  [{t['block_type']:2s}]  {t['name']:<22}  {t['description']}")

## 2. 按关键词检索（RAG 核心功能）

In [ ]:
# 示例1：检索「烧嘴控制」
results = search_templates('烧嘴控制', top_k=5)
print('检索「烧嘴控制」结果：')
for r in results:
    bar = '█' * int(r['score'] * 10) + '░' * (10 - int(r['score'] * 10))
    print(f"  [{r['block_type']:2s}] {r['name']:<20}  {r['score']:.2f}  {bar}")

In [ ]:
# 示例2：检索「报警 故障 复位」
results = search_templates('报警 故障 复位', top_k=3)
print('检索「报警 故障 复位」结果：')
for r in results:
    print(f"  [{r['block_type']:2s}] {r['name']:<20}  score={r['score']:.2f}")

In [ ]:
# 示例3：检索「PID 温度 调节」
results = search_templates('PID 温度 调节', top_k=3)
print('检索「PID 温度 调节」结果：')
for r in results:
    print(f"  [{r['block_type']:2s}] {r['name']:<20}  score={r['score']:.2f}")

## 3. 查看模板接口结构

In [ ]:
# 查看模板详情（截断摘要，约4000字符，节省上下文）
excerpt = get_template_xml('烧嘴控制', full=False)
if excerpt:
    print(f'模板「烧嘴控制」摘要（前500字符）：\n')
    print(excerpt[:500])
    print('\n...')
else:
    print('模板未找到')

In [ ]:
# 查看模板的接口变量（从 XML 中提取 Section 信息）
import xml.etree.ElementTree as ET

def show_interface(template_name: str):
    xml_str = get_template_xml(template_name, full=True)
    if not xml_str:
        print(f'未找到模板 {template_name}')
        return
    try:
        root = ET.fromstring(xml_str)
        print(f'=== 模板「{template_name}」接口变量 ===')
        # 搜索所有 Section 节点
        for section in root.iter():
            if section.tag.endswith('Section'):
                section_name = section.get('Name', '')
                members = [m for m in section if m.tag.endswith('Member')]
                if members:
                    print(f'\n  [{section_name}]')
                    for m in members:
                        name = m.get('Name', '')
                        dtype = m.get('Datatype', '')
                        print(f'    {name:<30} {dtype}')
    except ET.ParseError as e:
        print(f'XML 解析错误: {e}')

show_interface('烧嘴控制')

In [ ]:
show_interface('风机燃气')

## 4. 直接访问 TemplateLibrary 实例

In [ ]:
lib = TemplateLibrary.instance()

# 精确获取模板信息
info = lib.get('报警')
if info:
    print(f'名称: {info.name}')
    print(f'块类型: {info.block_type}')
    print(f'块名称: {info.block_name}')
    print(f'关键词: {info.keywords}')
    print(f'文件路径: {info.file_path}')

## 5. 扩充模板库

扩充步骤：
1. 在博途中完成梯形图，选中程序块 → 右键「导出」→ 选择 SimaticML 格式 → 保存 `.xml`
2. 将 `.xml` 文件复制到 `RAG/templates/` 目录
3. （可选）在 `rag_retriever.py` 的 `_TEMPLATE_META` 字典中补充关键词
4. 调用 `TemplateLibrary.reset()` 刷新缓存

In [ ]:
# 刷新模板库（添加新模板后调用）
TemplateLibrary.reset()
lib = TemplateLibrary.instance()
all_templates = lib.list_all()
print(f'刷新后共有 {len(all_templates)} 个模板')
for t in all_templates:
    print(f'  {t.display_name}')

## 6. 模拟 MCP Agent 调用流程

以下演示 Agent 在接到「生成烧嘴控制程序块」任务时的决策逻辑：

In [ ]:
def simulate_agent_decision(user_request: str):
    """模拟 Agent 决策：先检索模板，再决定是复用还是从零生成。"""
    print(f'用户需求: {user_request}')
    print('=' * 60)
    
    # Step 1: 检索模板
    print('\n[Step 1] 调用 search_plc_templates...')
    results = search_templates(user_request, top_k=3)
    
    if not results:
        print('  → 未找到相关模板，将使用 add_lad_block 从零生成')
        return
    
    best = results[0]
    print(f'  → 最佳匹配: [{best["block_type"]}] {best["name"]}  score={best["score"]:.2f}')
    
    # Step 2: 决策
    THRESHOLD = 0.5
    if best['score'] >= THRESHOLD:
        print(f'\n[Step 2] score >= {THRESHOLD}，调用 get_plc_template 查看接口...')
        excerpt = get_template_xml(best['name'], full=False)
        if excerpt:
            print(f'  → 获取到模板内容（{len(excerpt)} 字符），接口结构确认')
        print(f'\n[Step 3] 调用 import_template_block(template_name="{best["name"]}", device_name="PLC_1")')
        print(f'  → 模板直接导入，无需生成 XML ✅')
    else:
        print(f'\n[Step 2] 最高 score={best["score"]:.2f} < {THRESHOLD}，模板匹配度不足')
        print('  → 调用 add_lad_block(networks_json=<根据需求生成的梯级JSON>) 从零生成')


simulate_agent_decision('需要一个烧嘴温度控制程序块')

In [ ]:
simulate_agent_decision('电机正反转互锁控制')

## 7. 模板检索性能测试

In [ ]:
import time

queries = [
    '烧嘴控制',
    '风机燃气启停',
    '报警处理',
    'PID参数调节',
    '信号转换模拟量',
    '主程序OB1',
    '顺序控制步序',
]

print(f'{'查询词':<20} {'最佳匹配':<22} {'得分':>6}  耗时')
print('-' * 70)

for q in queries:
    t0 = time.perf_counter()
    results = search_templates(q, top_k=1)
    elapsed = (time.perf_counter() - t0) * 1000
    if results:
        r = results[0]
        print(f'{q:<20} [{r["block_type"]}] {r["name"]:<18} {r["score"]:>6.2f}  {elapsed:.1f}ms')
    else:
        print(f'{q:<20} (无匹配)                        {elapsed:.1f}ms')